# Plots and Figures

This notebook provides code for analyzing and visualizing results from the research paper. It includes:

1. **Dataset Analysis**
   - Counts of files, instances and class distributions across synthetic and real-world datasets
   - Analysis of failed clipping cases and filtering

2. **Model Performance Visualization** 
   - Detection error analysis (False Positives & False Negatives)
   - Visualization of model predictions vs ground truth
   - Training metrics plots (loss, learning rate, metrics over epochs)

3. **Data Visualization**
   - Sample visualization from synthetic datasets (ESRI & OSM)
   - Sample visualization from real-world datasets
   - Class Activation Map plots showing model feature activation

The code generates tables and figures that supplement the standard Ultralytics YOLO metrics and visualizations.


In [ ]:
base_path = "your"  # where you have the ds mounted/downloaded

#example structure for the Datasets changes require further adjustments below, e.g. when calling the exact splits
synthetic_ds_path = f"{base_path}/Datasets/synthetical/"
osm_ds_path = f"{base_path}/Datasets/merged/merged_osm_v1/"
ewi_ds_path = f"{base_path}/merged/merged_esri_v1/"

#in inches to be used for diffrent fomrats
single_column= 3.3
double_column = 7.0

import matplotlib.pyplot as plt

# set every figure to 400 dpi
plt.rcParams['figure.dpi'] = 200

In [ ]:
from collections import defaultdict
import os
from tqdm import tqdm

datasets = [synthetic_ds_path,
            osm_ds_path,
            ewi_ds_path]

for dataset in datasets:
    for split in ["train", "valid", "val", "test"]:
        try:
            labels_path = dataset + split + "/labels/"

            n_files = len(list(os.listdir(labels_path)))
            n_files_images = len(list(os.listdir(dataset + split + "/images/")))
            instances = 0
            colors = defaultdict(int)
            for filename in tqdm(os.listdir(labels_path)):
                with open(labels_path + filename, "r") as f:
                    lines = f.readlines()
                    instances += len(lines)
                    class_ids = [l.split()[0] for l in lines]
                    for class_id in class_ids:
                        colors[class_id] += 1
            print(dataset, split, instances, colors)
        except:
            pass


## The clipping wasnt successfull for some of the images, which we filtered out manually

In [ ]:
import os

for dataset in datasets:
    for split in ["train", "valid", "val", "test"]:
        try:
            labels_path = dataset + split + "/labels/"

            n_files = len(list(os.listdir(labels_path)))
            n_files_images = len(list(os.listdir(dataset + split + "/images/")))

            print(n_files_images, n_files)
        except:
            pass

## Find some FPs and FNs examples

this code section has been used to find examples for false positves and false negativews, based on predictions and ground trouths.

In [ ]:
import matplotlib.pyplot as plt
import os
import tifffile
from tqdm import tqdm
from ultralytics_MB import YOLO
import numpy as np


def iou(boxA, boxB):
    xa1, ya1, xa2, ya2 = boxA
    xb1, yb1, xb2, yb2 = boxB
    xi1, yi1 = max(xa1, xb1), max(ya1, yb1)
    xi2, yi2 = min(xa2, xb2), min(ya2, yb2)
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    areaA = (xa2 - xa1) * (ya2 - ya1)
    areaB = (xb2 - xb1) * (yb2 - yb1)
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0


iou_thresh = 0.5
results = {}

for background in ["osm", "esri"]:
    image_dir = (
        f"{base_path}"
        f"Datasets/merged/merged_{background}_v1/test/images" #adjust to match your datasets
    )
    image_files = [
        os.path.join(image_dir, f)
        for f in os.listdir(image_dir)
        if f.endswith(".tiff")
    ]
    model_path = f"{base_path}/runs/{background}/train/weights/best.pt" #Please replace with path to your model

    model = YOLO(model_path)
    fps, fns = [], []  # collect tuples for all images

    for img_path in tqdm(image_files, desc=f"Scanning {background}"):
        try:
            img = tifffile.imread(img_path)
            res = model.predict(img)
            pred_boxes = res[0].boxes.data.cpu().numpy()  # (N,6): x1,y1,x2,y2,conf,cls

            # load & decode GT
            label_path = img_path.replace("images", "labels").replace(".tiff", ".txt")
            with open(label_path) as f:
                lines = [l.strip() for l in f if l.strip()]

            h, w, _ = img.shape
            gt_boxes = []
            for line in lines:
                parts = line.split()
                if len(parts) == 5:
                    _, xc, yc, bw, bh = map(float, parts)
                    xc, yc, bw, bh = xc * w, yc * h, bw * w, bh * h
                    x1, y1 = xc - bw / 2, yc - bh / 2
                    x2, y2 = x1 + bw, y1 + bh
                else:
                    coords = list(map(float, parts[1:]))
                    xs, ys = coords[::2], coords[1::2]
                    x1, x2 = min(xs) * w, max(xs) * w
                    y1, y2 = min(ys) * h, max(ys) * h
                gt_boxes.append([x1, y1, x2, y2])

            # IoU matrix
            if gt_boxes and len(pred_boxes):
                mat = np.array([[iou(p[:4], g) for g in gt_boxes] for p in pred_boxes])
            else:
                mat = np.zeros((len(pred_boxes), len(gt_boxes)))

            matched_gt = set()
            this_fp = []
            for i, p in enumerate(pred_boxes):
                best = mat[i].max() if gt_boxes else 0
                if best < iou_thresh:
                    this_fp.append(p)
                else:
                    matched_gt.add(mat[i].argmax())

            this_fn = [g for idx, g in enumerate(gt_boxes) if idx not in matched_gt]

            if this_fp:
                fps.append((img, pred_boxes, lines))
            if this_fn:
                fns.append((img, pred_boxes, lines))
            print(len(fps), len(fns))
        except Exception as e:
            pass
    results[background] = {"fp": fps, "fn": fns}



## And lets plot them

We create plots, showing both the bboxes that belong to the ground trouth and those who belong to the predictions made by the model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from random import sample
import string

def parse_gt(gt_lines, w, h):
    """Convert GT strings to (x1,y1,x2,y2) in pixel coords."""
    boxes = []
    for line in gt_lines:
        parts = line.split()
        if len(parts) == 5:
            _, xc, yc, bw, bh = map(float, parts)
            xc, yc = xc * w, yc * h
            bw, bh = bw * w, bh * h
            x1, y1 = xc - bw/2, yc - bh/2
        else:
            coords = list(map(float, parts[1:]))
            xs, ys = coords[::2], coords[1::2]
            x1, y1 = min(xs) * w, min(ys) * h
            bw, bh = (max(xs)-min(xs)) * w, (max(ys)-min(ys)) * h
        boxes.append((x1, y1, x1 + bw, y1 + bh))
    return boxes

def compute_iou(b1, b2):
    """IoU between two (x1,y1,x2,y2) boxes."""
    xA, yA = max(b1[0], b2[0]), max(b1[1], b2[1])
    xB, yB = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    area1 = (b1[2]-b1[0])*(b1[3]-b1[1])
    area2 = (b2[2]-b2[0])*(b2[3]-b2[1])
    return inter / (area1 + area2 - inter + 1e-6)

def filter_duplicate_bboxes(r):
    """Remove duplicate GT lines."""
    img, pred_boxes, gt_lines = r
    seen = set(); unique_gt = []
    for line in gt_lines:
        if line not in seen:
            seen.add(line)
            unique_gt.append(line)
    return img, pred_boxes, unique_gt

# Deduplicate GTs
for bg in ["osm", "esri"]:
    results[bg]["fp"] = [filter_duplicate_bboxes(r) for r in results[bg]["fp"]]
    results[bg]["fn"] = [filter_duplicate_bboxes(r) for r in results[bg]["fn"]]

# specify exactly which IDs to include
allowed_ids = {
    "osm": {
        "fp": {13, 5, 4},
        "fn": {14, 6}
    },
    "esri": {
        "fp": {1, 5, 3},
        "fn": {19, 15}
    }
}

for _ in range(6):
    # 2 rows × 4 cols: 2 FP, 2 FN per background
    n_rows, n_cols = 2, 4
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(n_cols*5, n_rows*5), constrained_layout=True)
    axs = axs.flatten()

    for row, bg in enumerate(["osm", "esri"]):
        # build pools of (idx, record) only from allowed_ids
        fp_pool = [
            (i, r) for i, r in enumerate(results[bg]["fp"])
            if i in allowed_ids[bg]["fp"]
               and abs(len(r[1]) - len(r[2])) <= 5
        ]
        fn_pool = [
            (i, r) for i, r in enumerate(results[bg]["fn"])
            if i in allowed_ids[bg]["fn"]
               and abs(len(r[1]) - len(r[2])) <= 5
        ]

        # pick exactly 2 of each
        fp_samples = sample(fp_pool, 2)
        fn_samples = sample(fn_pool, 2)
        samples = fp_samples + fn_samples  # (sample_id, (img, preds, gts))

        for col in range(n_cols):
            ax = axs[row*n_cols + col]
            sample_id, (img, pred_boxes, gt_lines) = samples[col]
            h, w, _ = img.shape
            ax.imshow(img[..., [2,1,0]])  # BGR→RGB

            # parse GT and preds
            gt_boxes = parse_gt(gt_lines, w, h)
            preds = [(x1,y1,x2,y2,conf,cls) for x1,y1,x2,y2,conf,cls in pred_boxes]

            # match preds→GT
            matched = set(); tp, fp = [], []
            for p in preds:
                ious = [compute_iou(p, g) for g in gt_boxes]
                best = np.argmax(ious)
                if ious[best] >= 0.5:
                    tp.append(p); matched.add(best)
                else:
                    fp.append(p)
            fn = [gt_boxes[i] for i in range(len(gt_boxes)) if i not in matched]

            # draw the three types
            for x1,y1,x2,y2,conf,cls in tp:
                ax.add_patch(plt.Rectangle((x1,y1), x2-x1, y2-y1,
                                           linewidth=4, edgecolor='green',
                                           linestyle='-', facecolor='none'))
            for x1,y1,x2,y2,conf,cls in fp:
                ax.add_patch(plt.Rectangle((x1,y1), x2-x1, y2-y1,
                                           linewidth=4, edgecolor='blue',
                                           linestyle='--', facecolor='none'))
            for x1,y1,x2,y2 in fn:
                ax.add_patch(plt.Rectangle((x1,y1), x2-x1, y2-y1,
                                           linewidth=4, edgecolor='orange',
                                           linestyle=':', facecolor='none'))

            # determine letter & fault type
            idx = row * n_cols + col
            letter = string.ascii_lowercase[idx]
            fault = "FP" if col < 2 else "FN"

            # overlay the sample ID

            ax.set_title(r"$\mathbf{(%s)}$ %s %s" % (letter, bg.upper(), fault),
                         fontsize=11, pad=4)
            ax.axis('off')

    # shared legend
    legend_handles = [
        Line2D([0],[0], color='green', lw=2, linestyle='-', label='True Positive'),
        Line2D([0],[0], color='blue',  lw=2, linestyle='--',label='False Positive'),
        Line2D([0],[0], color='orange',lw=2, linestyle=':', label='False Negative'),
    ]
    fig.legend(handles=legend_handles, loc='lower right', ncol=3,
               frameon=True, fontsize=20, facecolor='white', edgecolor='black')

    plt.show()


In [ ]:
    #create table of len total files, how many contain a fp or fn
for bc in ["osm", "esri"]:
    n_files = len(
        os.listdir(f"{base_path}/merged/merged_{bc}_v1/test/images"))
    n_fp = len(results[bc]["fp"])
    n_fn = len(results[bc]["fn"])
    fp_set = set([r[0] for r in results[bc]["fp"]])
    fn_set = set([r[0] for r in results[bc]["fn"]])
    both = fp_set & fn_set
    print(f"{bc} n_files{n_files}, n fp = {n_fp}, n fn = {n_fn}, both = {len(both)}")

## Plot selected examples for the synthetic Dataset

Cherry Picked plots to show a good overview of common misstakes

In [ ]:
import tifffile
import matplotlib.pyplot as plt

#synthetic test dataset

files_osm = ["0_c7723c6c-662f-11ef-a633-1856806de7cd.tif", "e79da506-6f3c-11ef-b910-1856806de7cd.tif",
             "ce98f92a-8fee-11ef-a6ff-19b0dd293514.tif", "72cd7976-6cb3-11ef-a1d6-1856806de7cd.tif"]

files_esri = ["e64d94fa-6cf7-11ef-a38e-1856806de7cd.tif", "b0129bb6-6e31-11ef-ad43-1856806de7cd.tif",
              "41ff3dc0-6e3e-11ef-a867-1856806de7cd.tif", "b9886b82-6e3e-11ef-ad43-1856806de7cd.tif",
              "9736df4a-6e3a-11ef-ad43-1856806de7cd.tif"]

from pathlib import Path
import tifffile
import matplotlib.pyplot as plt

# base directory for your images
base_dir = Path(f"{base_path}/Datasets/synthetical/test/images")

groups = {"ESRI": files_esri[:4], "OSM": files_osm[:4]}
n_rows = len(groups)
n_cols = max(len(f) for f in groups.values())

single_column = 3.3  # inches, e.g.

fig, axs = plt.subplots(n_rows, n_cols,
                        figsize=(single_column, single_column*0.5),
                        squeeze=False)

# plot
for i, (label, fnames) in enumerate(groups.items()):
    for j, fname in enumerate(fnames):
        img = tifffile.imread(base_dir / fname)[:, :, [2,1,0]]
        ax = axs[i, j]
        ax.imshow(img)
        ax.axis("off")

# now set exactly 0.1" between plots
gap_in = 0.025
fig_w, fig_h = fig.get_size_inches()
fig.subplots_adjust(
    wspace = gap_in / (fig_w / n_cols),
    hspace = gap_in / (fig_h / n_rows),
    left=0, right=1, top=1, bottom=0,
)

plt.savefig("exampley_synthetical_imagery_no_title.png", bbox_inches='tight', pad_inches=0)
plt.show()
plt.close(fig)

In [ ]:
import matplotlib.pyplot
from pathlib import Path
from random import sample
import tifffile
import matplotlib.pyplot as plt
import os

base_dir_osm = Path(f"{base_path}/Datasets/merged/merged_osm_v1/test/images")
base_dir_esri = Path(f"{base_path}/Datasets/merged/merged_esri_v1/test/images")

# group names and (sliced) file lists
groups = {
    "ESRI": [base_dir_esri / f for f in sample(os.listdir(base_dir_esri), 4)],
    "OSM": [base_dir_osm / f for f in sample(os.listdir(base_dir_osm), 4)],
}

n_rows = len(groups)
n_cols = max(len(f) for f in groups.values())

single_column = 3.3  # inches, e.g.

fig, axs = plt.subplots(n_rows, n_cols,
                        figsize=(single_column, single_column*0.5),
                        squeeze=False)

# plot
for i, (label, fnames) in enumerate(groups.items()):
    for j, fname in enumerate(fnames):
        img = tifffile.imread(fname)[:, :, [2,1,0]]
        ax = axs[i, j]
        ax.imshow(img)
        ax.axis("off")

# now set exactly 0.1" between plots
gap_in = 0.025
fig_w, fig_h = fig.get_size_inches()
fig.subplots_adjust(
    wspace = gap_in / (fig_w / n_cols),
    hspace = gap_in / (fig_h / n_rows),
    left=0, right=1, top=1, bottom=0,
)

plt.savefig("exampley_realworld_imagery_no_title.png", bbox_inches='tight', pad_inches=0)
plt.show()
plt.close(fig)


## Visualize Training Results

This cell processes and visualizes the training results from a CSV file that contains per-epoch statistics. The data includes:

- Training and validation loss metrics
- Learning rate progression
- Other model performance metrics (e.g., mAP, precision, recall)

A results.csv file is generated during model training and contains epoch-wise performance statistics that help track the training progress and model convergence.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --- Configuration ---
base_path = "/home/clemens/mnt/sds-hd/sd17f001/sketch-map-tool"
files = [
    ("Synthetic", f"{base_path}/runs/synthetic/train/results.csv"),
    ("OSM",       f"{base_path}/runs/osm/train/results.csv"),
    ("EWI",      f"{base_path}/runs/esri/train/results.csv"),
]
single_column = 2.4  # inches

n = len(files)
fig, axs = plt.subplots(
    nrows=n,
    ncols=3,
    figsize=(single_column * 4, single_column * 0.33 * n * 5),
    sharey='col',
    gridspec_kw={
        'wspace': 0.25,
        'hspace': 0.33,
        'top':   0.90,
        'bottom':0.2,  # make extra room for legends
        'left':  0.05
    }
)
plt.style.use('ggplot')
cmap = plt.get_cmap('tab10')

for row, (name, path) in enumerate(files):
    df = pd.read_csv(path)
    x = df['epoch'] if 'epoch' in df else df.index

    train_loss = df.filter(regex=r'^train/.*_loss$')
    val_loss   = df.filter(regex=r'^val/.*_loss$')
    lr_cols    = df.filter(regex=r'^lr/').columns
    metrics    = df.filter(regex=r'^metrics/').columns

    # --- Loss (col 0) ---
    ax0 = axs[row, 0]
    ax0.set_facecolor('#f7f7f7')
    for i, col in enumerate(train_loss.columns):
        ax0.plot(x, train_loss[col], marker='o',
                 markevery=max(len(x)//10,1), color=cmap(i), linewidth=1, label=col)
    for i, col in enumerate(val_loss.columns, start=len(train_loss.columns)):
        ax0.plot(x, val_loss[col], linestyle='--', marker='s',
                 markevery=max(len(x)//10,1), color=cmap(i), alpha=0.7,
                 linewidth=1, label=col)
    ax0.set_yscale('log')
    all_l = np.hstack([train_loss.values.ravel(), val_loss.values.ravel()])
    emin, emax = int(np.floor(np.log10(all_l.min()))), int(np.ceil(np.log10(all_l.max())))
    decades = [10.**i for i in range(emin, emax+1)]
    ticks = sorted(decades + ([0.5] if 0.5>=decades[0] and 0.5<=decades[-1] else []))
    ax0.set_ylim(decades[0], all_l.max()*1.1)
    ax0.set_yticks(ticks)
    ax0.set_yticklabels([f"{t:g}" for t in ticks])
    if row == 0:
        ax0.set_title("Loss",fontsize=10)
    ax0.set_ylabel(
        name,
        rotation=90,
        labelpad=10,
        fontsize=10,
        fontweight='bold',
        va='center',
        ha='right'
    )

    # --- Learning Rate (col 1) ---
    ax1 = axs[row, 1]
    ax1.set_facecolor('#f7f7f7')
    for i, col in enumerate(lr_cols):
        ax1.plot(x, df[col], marker='^',
                 markevery=max(len(x)//10,1), color=cmap(i), linewidth=1, label=col)
    if row == 0:
        ax1.set_title("Learning Rate",fontsize=10)
    ax1.grid(True, linestyle=':')
    if row == n-1:
        ax1.set_xlabel("Epoch")

    # --- Metrics (col 2) ---
    ax2 = axs[row, 2]
    ax2.set_facecolor('#f7f7f7')
    for i, col in enumerate(metrics):
        ax2.plot(x, df[col], marker='D',
                 markevery=max(len(x)//10,1), color=cmap(i), linewidth=1, label=col)
    if row == 0:
        ax2.set_title("Metrics",fontsize=10)
    ax2.grid(True, linestyle=':')

# Align the row‐labels
fig.align_ylabels(axs[:, 0])

# Place one legend per column, matching each column width
legend_height = 0.025   # in figure coords
legend_y = 0.10        # vertical position in figure coords (just above bottom)
for col in range(3):
    ax = axs[0, col]
    handles, labels = ax.get_legend_handles_labels()
    bbox = ax.get_position()

    fig.legend(
        handles, labels,
        loc='upper left',
        bbox_to_anchor=(bbox.x0- bbox.width*0.15, legend_y, bbox.width*1.3, legend_height * 2),  # Enlarged height
        mode='expand',
        ncol=min(len(labels), 2) if col != 2 else 1,
        fontsize=11,           # Increased font size
        frameon=True,
        borderpad=0.1,        # More space inside the legend box
        labelspacing=1,     # More spacing between labels
        handlelength=1.5      # Slightly longer line symbols
    )
plt.tight_layout()
plt.show()




## Generate a series of Class Activation Map Plots

In the following code creasts Class Activation Map (CAMs) Plots, that highlighlight

In [ ]:
from ultralytics_MB import YOLO
import os
import cv2
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import tifffile
from random import sample
from matplotlib.colors import TwoSlopeNorm

plt.rcParams['figure.dpi'] = 200

split = "test"
bc = "osm"

model_path = f"{base_path}/runs/{bc}/train/weights/best.pt"
model = YOLO(model_path)

images = f"{base_path}/Datasets/merged/merged_{bc}_v1/{split}/images"
paths = [os.path.join(images, f) for f in os.listdir(images) if ".tif" in f]

feature_maps = {}
def hook_fn(name):
    def hook(module, input, output):
        feature_maps[name] = output.detach()
    return hook

layer_indices = [30, 60, 26, 56, 23, 53, 61, 62, 63, 69, 72, 75]
for i in layer_indices:
    model.model.model[i].register_forward_hook(hook_fn(f"layer_{i}"))

def get_raw_heatmap(fmap, size):
    fmap2d = fmap.squeeze(0).mean(dim=0, keepdim=True)
    up = F.interpolate(fmap2d.unsqueeze(0),
                       size=size, mode='bilinear', align_corners=False)
    return up.squeeze().cpu().numpy()

def plot_activation(ax, fmap, base_img, title=None, vmin=None, vmax=None,shift=0.0, alpha=0.7):
    fmap2d = fmap.squeeze(0).mean(dim=0, keepdim=True)
    up = F.interpolate(fmap2d.unsqueeze(0),
                       size=base_img.shape[:2], mode='bilinear', align_corners=False)
    heat = up.squeeze().cpu().numpy()

    if vmin is None or vmax is None:
        lim = max(abs(heat.min()), abs(heat.max()))
    else:
        lim = max(abs(vmin), abs(vmax))
    vmin_plot, vmax_plot = -lim, lim
    norm = TwoSlopeNorm(vmin=vmin_plot, vcenter=0, vmax=vmax_plot)
    cmap = plt.get_cmap('bwr')

    ax.imshow(base_img)
    print(norm)

    ax.imshow(heat+shift, cmap=cmap, norm=norm, alpha=alpha)
    if title:
        ax.set_title(title)
    ax.axis('off')
    return cmap, norm  # return so we know what mappable to grab

for path in sample(paths, 10):
    img = tifffile.imread(path)
    res = model.predict(img)[0]

    # prepare images
    if img.ndim == 2:
        img_rgb = np.stack([img] * 3, axis=-1)
    elif img.ndim == 3 and img.shape[0] in [1,3]:
        img_rgb = np.moveaxis(img, 0, -1)
    else:
        img_rgb = img[..., [2,1,0]].copy()
    img_rgb = (img_rgb - img_rgb.min()) / (img_rgb.max() - img_rgb.min() + 1e-6)
    img_cir = img[..., [5,4,3]].copy()
    img_cir = (img_cir - img_cir.min()) / (img_cir.max() - img_cir.min() + 1e-6)

    h30 = get_raw_heatmap(feature_maps["layer_30"],img_rgb.shape[:2])
    h60 = get_raw_heatmap(feature_maps["layer_60"],img_rgb.shape[:2])
    h61 = get_raw_heatmap(feature_maps['layer_61'], img_rgb.shape[:2])#
    h62 = get_raw_heatmap(feature_maps['layer_62'], img_rgb.shape[:2])
    h63 = get_raw_heatmap(feature_maps['layer_63'], img_rgb.shape[:2])

    global_min = min(h30.min(), h60.min(), h61.min())
    global_max = max(h30.max(), h60.max(), h61.max())

    # --- First Figure with titles ---
    fig, axs = plt.subplots(1, 6, figsize=(20, 6))

    axs[0].imshow(img_rgb); axs[0].set_title("Input Image - Marked"); axs[0].axis('off')
    fig.text(-0.005, 0.5, "EWI" if bc.lower() == "esri" else "OSM",
         va='center', rotation='vertical', fontsize=12, weight='bold')
    axs[1].imshow(img_cir); axs[1].set_title("Reference Map"); axs[1].axis('off')

    # plot overlays and capture cmap/norm from the first overlay
    _, _ = plot_activation(axs[2], feature_maps['layer_30'], img_rgb,
                          "Layer 30 Activation", vmin=global_min, vmax=global_max)
    _, _ = plot_activation(axs[3], feature_maps['layer_60'], img_cir,
                          "Layer 60 Activation", vmin=global_min, vmax=global_max)
    cmap, norm = plot_activation(axs[4], feature_maps['layer_61'], img_rgb,
                                 "Layer 61 Activation", vmin=global_min, vmax=global_max)

    # bounding boxes
    axs[5].imshow(img_rgb)
    for conf, box in zip(res.boxes.conf.cpu().numpy(), res.boxes.xyxy.cpu().numpy()):
        x1, y1, x2, y2 = box
        axs[5].add_patch(plt.Rectangle((x1,y1), x2-x1, y2-y1, linewidth=2,
                                       edgecolor='lime', facecolor='none'))
        axs[5].text(x1, y1-5, f"{conf:.2f}", color='white', fontsize=8,
                    bbox=dict(facecolor='black', alpha=0.5, pad=1))
    axs[5].set_title("Predicted Bounding Boxes"); axs[5].axis('off')

    # add colorbar on the right
    fig.subplots_adjust(right=0.88)  # make room
    cbar_ax = fig.add_axes([1.0, 0.25, 0.02, 0.5])  # [left, bottom, width, height]
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])  # no data—mappable only
    fig.colorbar(sm, cax=cbar_ax)

    plt.tight_layout()
    plt.show()

    # --- First Figure with titles ---
    fig, axs = plt.subplots(1, 6, figsize=(20, 6))

    axs[0].imshow(img_rgb); axs[0].axis('off')
    fig.text(-0.005, 0.5, "EWI" if bc.lower() == "esri" else "OSM",
         va='center', rotation='vertical', fontsize=12, weight='bold')
    axs[1].imshow(img_cir); axs[1].axis('off')

    # plot overlays and capture cmap/norm from the first overlay
    _, _ = plot_activation(axs[2], feature_maps['layer_30'], img_rgb,
                          None, vmin=global_min, vmax=global_max)
    _, _ = plot_activation(axs[3], feature_maps['layer_60'], img_cir,
                          None, vmin=global_min, vmax=global_max)
    cmap, norm = plot_activation(axs[4], feature_maps['layer_61'], img_rgb,
                        None, vmin=global_min, vmax=global_max)

    # bounding boxes
    axs[5].imshow(img_rgb)
    for conf, box in zip(res.boxes.conf.cpu().numpy(), res.boxes.xyxy.cpu().numpy()):
        x1, y1, x2, y2 = box
        axs[5].add_patch(plt.Rectangle((x1,y1), x2-x1, y2-y1, linewidth=2,
                                       edgecolor='lime', facecolor='none'))
        axs[5].text(x1, y1-5, f"{conf:.2f}", color='white', fontsize=8,
                    bbox=dict(facecolor='black', alpha=0.5, pad=1))
    axs[5].axis('off')

    # add colorbar on the right
    fig.subplots_adjust(right=0.88)  # make room
    cbar_ax = fig.add_axes([1.0, 0.25, 0.02, 0.5])  # [left, bottom, width, height]
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])  # no data—mappable only
    fig.colorbar(sm, cax=cbar_ax)

    plt.tight_layout()